### Armado del avión

Inicializado con 25 filas, cada fila tiene cuatro lugares disponibles: dos lugares que dan a la ventana (izquierda y derecha) y dos lugares que dan al pasillo.

In [2]:
import random

n_filas = 25
lados = [-2, -1, 1, 2]
# -2 es ventana izquierda | -1 es pasillo izquierda | 1 es pasillo derecha | 2 es ventana derecha

# Creamos los asientos fila por fila, respetando el orden de los lados.
asientos = []

for fila in range(1, n_filas + 1):
    for lado in lados:
        asiento = (fila, lado)
        asientos.append(asiento)

# Para corroborar que esté bien
print(f"Cantidad de asientos: {len(asientos)}") # debería dar 25x4 = 100
print("Primeros 8 asientos:", asientos[:8])

Cantidad de asientos: 100
Primeros 8 asientos: [(1, -2), (1, -1), (1, 1), (1, 2), (2, -2), (2, -1), (2, 1), (2, 2)]


### Modelado de políticas

En este trabajo vamos a estudiar las políticas de llenado del avión random, back-to-front, WILMA y Steffen.

In [3]:
# Politica Random
def orden_random(asientos):
    """
    Politica Random: no hay ningun orden. Cada pasajero sube en
    cualquier momento, sin importar fila ni asiento.
    """
    # Orden en el que suben los pasajeros. Hasta acá es en orden
    #   la cola es la gente haciendo la cola para subirse al avion
    cola = asientos.copy()
    # Mezclamos 
    random.shuffle(cola)
    return cola


# Politica Back-to-Front
def orden_back_to_front(asientos, n_bloques=5):
    """
    Back-to-Front: se llena de atras hacia adelante, en bloques de filas,
    con orden random adentro de cada bloque.

    Con n_filas=25 y n_bloques=5, cada bloque tiene 25/5 = 5 filas (elegimos 5 bloques así cada bloque tiene la misma cantidad de asientos)

    """
    filas_por_bloque = n_filas // n_bloques

    cola = []

    # range(n_bloques - 1, -1, -1): range(start, stop, step) con step negativo cuenta para atrás
    # Usamos el indice de bloque mas alto (el de las ultimas filas) primero, porque Back-to-Front arranca por atras.
    for bloque in range(n_bloques - 1, -1, -1):
        fila_inicio = bloque * filas_por_bloque + 1     # La fila que cierra el grupo del bloque actual
        fila_final = fila_inicio + filas_por_bloque - 1 # La fila que comienza el grupo del bloque actual

        asientos_bloque = []

        for asiento in asientos:
            fila = asiento[0]   # numero de fila

            if fila_inicio <= fila <= fila_final: # Si estas en una fila del bloque actual
                asientos_bloque.append(asiento)   # Sos parte de los asientos del bloque actual


        # desordenamos dentro del bloque actual (los otros bloques no se tocan).
        random.shuffle(asientos_bloque)

        # .extend() agrega todos los elementos de asientos_bloque al final
        cola.extend(asientos_bloque)

    return cola



# Politica Wilma (Window-Middle-Aisle, pero nosotros no tenemos asiento del medio, asi que es Window-Aisle)

def orden_wilma(asientos):
    """
    WILMA: por tipo de asiento en todo el avion a la vez, no por fila.
    En este avion no hay asiento del medio, asi que se reduce a ventanilla o pasillo
    abs(lado) == 2 -> ventana (lados -2 y 2)
    abs(lado) == 1 -> pasillo (lados -1 y 1)
    """
    ventanas = []
    pasillos = []

    for asiento in asientos:
        lado = asiento[1]

        if abs(lado) == 2:
            ventanas.append(asiento)
        else:
            pasillos.append(asiento)

    random.shuffle(ventanas)
    random.shuffle(pasillos)

    return ventanas + pasillos


# Politica Steffen (sin asiento del medio)

def orden_steffen(asientos):
    """
    Steffen: ventana filas impares -> ventana filas pares -> pasillo filas impares -> pasillo filas pares.
    """
    
    ventana_impar = []
    ventana_par = []
    pasillo_impar = []
    pasillo_par = []

    for asiento in asientos:
        fila = asiento[0]
        lado = asiento[1]

        # Primero distinguimos ventana o pasillo; luego, fila impar o par.
        if abs(lado) == 2:
            if fila % 2 == 1:
                ventana_impar.append(asiento)
            else:
                ventana_par.append(asiento)
        else:
            if fila % 2 == 1:
                pasillo_impar.append(asiento)
            else:
                pasillo_par.append(asiento)

    random.shuffle(ventana_impar)
    random.shuffle(ventana_par)
    random.shuffle(pasillo_impar)
    random.shuffle(pasillo_par)

    return ventana_impar + ventana_par + pasillo_impar + pasillo_par

In [6]:
cola_random = orden_random(asientos)
print("\nPrimeros 10 pasajeros en politica Random:")
print(cola_random[:10])

cola_btf = orden_back_to_front(asientos, n_bloques=5)
print("\nPrimeros 10 pasajeros en politica Back-to-Front (5 bloques de 5 filas):")
print(cola_btf[:10])
print("Filas de esos primeros 10 (deberian ser todas 21-25, en algun orden):")
filas_primeros_pasajeros = []

for asiento in cola_btf[:10]:
    fila = asiento[0]
    filas_primeros_pasajeros.append(fila)

print(filas_primeros_pasajeros)

cola_wilma = orden_wilma(asientos)
print("\nPrimeros 10 pasajeros en politica WILMA (deberian ser todos ventana, lado -2 o 2):")
print(cola_wilma[:10])

cola_steffen = orden_steffen(asientos)
print("\nPrimeros 10 pasajeros en politica Steffen (ventana, filas impares primero):")
print(cola_steffen[:10])


Primeros 10 pasajeros en politica Random:
[(25, 2), (4, 1), (18, -2), (19, 1), (23, 2), (17, -1), (12, 1), (11, 2), (1, 2), (14, -1)]

Primeros 10 pasajeros en politica Back-to-Front (5 bloques de 5 filas):
[(21, -2), (21, 2), (22, -1), (23, 1), (24, -2), (21, -1), (25, 2), (22, 1), (25, 1), (24, -1)]
Filas de esos primeros 10 (deberian ser todas 21-25, en algun orden):
[21, 21, 22, 23, 24, 21, 25, 22, 25, 24]

Primeros 10 pasajeros en politica WILMA (deberian ser todos ventana, lado -2 o 2):
[(4, 2), (16, -2), (13, -2), (20, 2), (2, -2), (12, 2), (11, 2), (10, -2), (18, -2), (1, 2)]

Primeros 10 pasajeros en politica Steffen (ventana, filas impares primero):
[(19, -2), (25, 2), (9, 2), (3, 2), (21, 2), (5, 2), (11, 2), (7, -2), (7, 2), (13, 2)]


# Simulación

reloj

estructura:
donde esta cada pasajero
que caracteristicas tiene cada pasajero
que sucede en cada espacio del avion

"""
Ubicación en el avión (en qué fila está parado o sentado).
Destino: su fila y lado asignado (ventana o pasillo).
Si tiene carry-on o no (se sortea con probabilidad p, dato del problema).
Velocidad de movimiento (depende de si tiene carry-on: 3 seg/fila sin carry-on, 6 seg/fila con carry-on — esto ya lo da la consigna).
Estado: sentado / caminando / bloqueado-esperando / guardando carry-on / sentándose.
"""

1. Creamos clase pasajero. Para cada tupla-asiento se inicializa un pasajero.

In [ ]:
def carry_on (p):
    numero_aleatorio = random.random()

    if numero_aleatorio < p:
        tiene_carry_on = 1
    else:
        tiene_carry_on = 0

    return tiene_carry_on

"""
Los estados posibles son:
    0 : Caminando
    1 : Sentado
    2 : Bloqueado-esperando
    3 : Guardando carry-on
    4 : Sentandose
    5 : Parado/a
"""

class Pasajero:
    def __init__(self, asiento, id, p_carryon=0.5):
        # Guardamos los datos de cada pasajero.
        self.asiento = asiento
        self.ubicacion = [-1]  # No está adentro del avión
        self.carryon = carry_on(p_carryon)
        self.velocidad = 3*(self.carryon+1)
        self.estado = [0]
        self.id = id


    def __repr__(self):
        return (
            f"Pasajero ( con asiento = {self.asiento}, "
            f"tiene_carry_on={self.carryon}), "
            f"y está en estado = {self.estado} )"
        )

    def actualizar_estado():
        pass

    def tiempo_sentado():
        pass
    def tiempo_dejar_pasar():
        pass

    def tiempo_guardado_carryon():
        pass

In [ ]:
import numpy as np

class Avion:
    def __init__(self, pasajeros):
        self.n_filas = 25
        self.n_columnas = 5
        self.capacidad = self.n_filas * 4
        self.pasajeros = pasajeros

        # Columnas: ventana izquierda, asiento de pasillo izquierdo,
        # pasillo central, asiento de pasillo derecho y ventana derecha.
        # Las filas de NumPy van de 0 a 24; las del avion, de 1 a 25.
        # Cada pasajero debe tener un ID positivo y unico.
        # Su ID aparece en una sola posicion de la matriz mientras esta a bordo.
        self.matriz = np.zeros((self.n_filas, self.n_columnas), dtype=int)

    def cantidad_pasajeros(self):
        """Cuenta los pasajeros a bordo, incluyendo los que estan en el pasillo."""
        posiciones_ocupadas = self.matriz > 0
        cantidad = np.count_nonzero(posiciones_ocupadas)
        return cantidad

    def cantidad_sentados(self):
        """Cuenta a los pasajeros a bordo cuyo ultimo estado es sentado (1)."""
        cantidad = 0

        for pasajero in self.pasajeros:
            posiciones_del_pasajero = self.matriz == pasajero.id
            esta_en_el_avion = np.any(posiciones_del_pasajero)
            estado_actual = pasajero.estado[-1]

            if esta_en_el_avion and estado_actual == 1:
                cantidad += 1

        return cantidad

    def esta_lleno(self):
        """El embarque esta completo cuando los 100 pasajeros estan sentados."""
        pasajeros_sentados = self.cantidad_sentados()
        return pasajeros_sentados == self.capacidad
